In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PREPROCESSED_PATH = Path(
    r"C:\Users\PC\IELTS-Writing-Evals\dataset\asap\training_set_rel3_preprocessed.csv"
)
PROMPT2_TRAITS_PATH = Path(
    r"C:\Users\PC\IELTS-Writing-Evals\dataset\asap++\Prompt-2.csv"
)

OUTPUT_PATH = Path(
    r"C:\Users\PC\IELTS-Writing-Evals\dataset\asap++\Prompt-2-concat.csv"
)


In [3]:
def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    raise ValueError(f"Unsupported file type: {path.suffix}")


In [4]:
pre = read_table(PREPROCESSED_PATH)
p2_traits = read_table(PROMPT2_TRAITS_PATH)

print("Preprocessed columns:")
print(pre.columns.tolist())

print("\nPrompt 2 trait columns:")
print(p2_traits.columns.tolist())

Preprocessed columns:
['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm']

Prompt 2 trait columns:
['Essay ID', 'Content', 'Organization', 'Word Choice', 'Sentence Fluency', 'Conventions']


In [5]:
# =========================
# 4. Chuẩn hóa tên cột
# =========================

# Chuẩn hóa preprocessed
pre = pre.rename(columns={
    "Essay ID": "essay_id",
    "essay_set": "essay_set",
    "essay": "essay",
    "domain1_score": "domain1_score",
    "domain1_score_norm": "domain1_score_norm",
})

# Chuẩn hóa trait file prompt 2
# Script này cover cả trường hợp tên cột bị cắt như trong screenshot:
# Essay ID, Content, Organizat, Word Cho, Sentence, Conventions
p2_traits = p2_traits.rename(columns={
    "Essay ID": "essay_id",
    "essay_id": "essay_id",

    "Content": "p2_content",
    "Ideas & Content": "p2_content",
    "Ideas and Content": "p2_content",

    "Organizat": "p2_organization",
    "Organization": "p2_organization",

    "Word Cho": "p2_word_choice",
    "Word Choice": "p2_word_choice",

    "Sentence": "p2_sentence_fluency",
    "Sentence Fluency": "p2_sentence_fluency",

    "Conventions": "p2_conventions",
})


In [6]:
# =========================
# 5. Check cột bắt buộc
# =========================

required_pre_cols = [
    "essay_id",
    "essay_set",
    "essay",
    "domain1_score",
    "domain1_score_norm",
]

required_trait_cols = [
    "essay_id",
    "p2_content",
    "p2_organization",
    "p2_word_choice",
    "p2_sentence_fluency",
    "p2_conventions",
]

missing_pre = [c for c in required_pre_cols if c not in pre.columns]
missing_traits = [c for c in required_trait_cols if c not in p2_traits.columns]

if missing_pre:
    raise ValueError(f"Missing columns in preprocessed file: {missing_pre}")

if missing_traits:
    raise ValueError(f"Missing columns in prompt 2 trait file: {missing_traits}")

In [7]:
# =========================
# 6. Lọc prompt 2 từ preprocessed
# =========================

pre_p2 = pre.loc[pre["essay_set"] == 2, required_pre_cols].copy()

# Đảm bảo essay_id cùng kiểu dữ liệu để merge không lỗi
pre_p2["essay_id"] = pre_p2["essay_id"].astype(int)
p2_traits["essay_id"] = p2_traits["essay_id"].astype(int)

In [8]:

# =========================
# 7. Chỉ giữ cột trait cần dùng
# =========================

p2_traits = p2_traits[required_trait_cols].copy()


In [10]:
# Kiểm tra duplicate trong file trait prompt 2
dup_traits = p2_traits[p2_traits["essay_id"].duplicated(keep=False)].sort_values("essay_id")

print("Số dòng bị trùng essay_id trong p2_traits:", len(dup_traits))
print("Số essay_id bị trùng:", dup_traits["essay_id"].nunique())

dup_traits.head(20)

Số dòng bị trùng essay_id trong p2_traits: 2
Số essay_id bị trùng: 1


,essay_id,p2_content,p2_organization,p2_word_choice,p2_sentence_fluency,p2_conventions
1377,4355,6,4,6,5,6
1378,4355,6,6,6,6,6


In [11]:
trait_cols = [
    "p2_content",
    "p2_organization",
    "p2_word_choice",
    "p2_sentence_fluency",
    "p2_conventions",
]

# Gom các essay_id bị trùng bằng trung bình
p2_traits = (
    p2_traits
    .groupby("essay_id", as_index=False)[trait_cols]
    .mean()
)

# Round về int vì trait score là thang điểm rời rạc
for col in trait_cols:
    p2_traits[col] = p2_traits[col].round().astype(int)

# Kiểm tra lại
print("Duplicate essay_id còn lại:", p2_traits["essay_id"].duplicated().sum())

Duplicate essay_id còn lại: 0


In [12]:
benchmark_p2 = pre_p2.merge(
    p2_traits,
    on="essay_id",
    how="inner",
    validate="one_to_one"
)

print("Rows after merge:", len(benchmark_p2))

Rows after merge: 1799


In [13]:
# =========================
# 9. Drop NaN ở trait score
# =========================

trait_cols = [
    "p2_content",
    "p2_organization",
    "p2_word_choice",
    "p2_sentence_fluency",
    "p2_conventions",
]

benchmark_p2 = benchmark_p2.dropna(subset=trait_cols).copy()

In [14]:
# =========================
# 10. Thêm prompt_id cho dễ benchmark
# =========================

benchmark_p2.insert(
    loc=2,
    column="prompt_id",
    value=2
)

In [15]:
# =========================
# 11. Ép kiểu score trait thành int nếu được
# =========================

for col in trait_cols:
    benchmark_p2[col] = benchmark_p2[col].astype(int)

In [16]:
# =========================
# 12. Lưu file benchmark prompt 2
# =========================

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
benchmark_p2.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("\nDone.")
print(f"Prompt 2 rows in preprocessed: {len(pre_p2)}")
print(f"Prompt 2 rows after merge/dropna: {len(benchmark_p2)}")
print(f"Saved to: {OUTPUT_PATH}")

print("\nFinal columns:")
print(benchmark_p2.columns.tolist())


Done.
Prompt 2 rows in preprocessed: 1800
Prompt 2 rows after merge/dropna: 1799
Saved to: C:\Users\PC\IELTS-Writing-Evals\dataset\asap++\Prompt-2-concat.csv

Final columns:
['essay_id', 'essay_set', 'prompt_id', 'essay', 'domain1_score', 'domain1_score_norm', 'p2_content', 'p2_organization', 'p2_word_choice', 'p2_sentence_fluency', 'p2_conventions']
